# Capa Oro: Datos listos para Machine Learning y Minería de Datos

En este notebook realizamos la construcción de la **Capa Oro** del sistema de telemetría de controladores Epever.

**Objetivos:**
- Integrar y depurar los datos enriquecidos de la Capa Plata.
- Generar **variables derivadas** (feature engineering) relevantes para paneles solares y baterías.
- Validar la calidad final de los datos y mostrar distribuciones visuales.
- Exportar los datasets definitivos para el entrenamiento de modelos.

## Importar las librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.simplefilter(action='ignore')

PLATA_DIR = Path('../data/plata')
ORO_DIR = Path('../data/oro')
ORO_DIR.mkdir(parents=True, exist_ok=True)

# Archivos de entrada (generados en la notebook de enriquecimiento)
archivo_diario = PLATA_DIR / 'dataset_telemetria_diario_enriquecido.csv'
archivo_base = PLATA_DIR / 'dataset_telemetria_base_enriquecido.csv'

print("Configuración y directorios listos.")

## Carga de Datos desde Capa Plata

In [ ]:
# Lectura de datasets
df_diario = pd.read_csv(archivo_diario, parse_dates=['FECHA'])
df_base = pd.read_csv(archivo_base, parse_dates=['timestamp'])

print('Datos cargados:')
print(' - Diario (agrupado):', df_diario.shape)
print(' - Base (frecuencia 10 min):', df_base.shape)

display(df_diario.head())


## Generación de Variables Derivadas (Feature Engineering)

Creamos nuevas variables (features) que aportan un contexto mucho más rico para minería de datos, como las caídas de voltaje de la batería o el rango de trabajo de los paneles.

In [ ]:
if not df_diario.empty:
    # Rango de voltaje de batería en el día (ayuda a ver la salud de la batería)
    df_diario['battery_voltage_RANGE'] = df_diario['battery_voltage_V_MAX'] - df_diario['battery_voltage_V_MIN']
    
    # Eficiencia de captura térmica (rango de temperatura del controlador)
    df_diario['controller_temp_RANGE'] = df_diario['controller_temp_C_MAX'] - df_diario['controller_temp_C_MEAN']
    
    # Clasificación de día (Productivo vs Nublado/Bajo desempeño) basado en la energía generada.
    # Para hacerlo por estación, sacamos la media de energía de esa estación.
    df_diario['es_dia_productivo'] = df_diario.groupby('station_name')['energy_generated_today_kWh_MAX'].transform(lambda x: x >= x.median())
    
    # Redondeo para estética y limpieza
    cols_deri = ['battery_voltage_RANGE', 'controller_temp_RANGE']
    df_diario[cols_deri] = df_diario[cols_deri].round(2)
    
    print("Nuevas variables derivadas creadas:")
    display(df_diario[['station_name', 'FECHA', 'battery_voltage_RANGE', 'controller_temp_RANGE', 'es_dia_productivo']].head())


## Validación Final de la Capa Oro
Confirmamos que ya no existen nulos y validamos la continuidad de la serie temporal por estación.

In [ ]:
if not df_diario.empty:
    nulos_restantes = df_diario.isnull().sum().sum()
    print(f'Total de valores nulos en Capa Oro: {nulos_restantes}')
    
    # Confirmar fechas continuas
    fechas_esperadas = pd.date_range(df_diario['FECHA'].min(), df_diario['FECHA'].max(), freq='D').date
    fechas_existentes = set(df_diario['FECHA'].dt.date.unique())
    fechas_faltantes = sorted(set(fechas_esperadas) - fechas_existentes)
    
    print("\nFechas globales faltantes detectadas en el sistema general:")
    if not fechas_faltantes:
        print(" - Ninguna. La serie temporal diaria es continua.")
    else:
        print(fechas_faltantes)


## Visualizaciones Analíticas
Las visualizaciones nos ayudan a entender la calidad del dataset y la distribución de nuestras variables derivadas antes de inyectarlas a modelos.

### 1. Evolución del Nivel de Batería (SOC) y Generación por Estación

In [ ]:
if not df_diario.empty:
    variables_plot = ['battery_soc_%_MEAN', 'energy_generated_today_kWh_MAX']
    nombres_plot = ['Nivel Medio de Batería (%)', 'Energía Diaria Generada (kWh)']
    
    for var, name in zip(variables_plot, nombres_plot):
        plt.figure(figsize=(14, 5))
        sns.lineplot(data=df_diario, x='FECHA', y=var, hue='station_name', marker='o')
        plt.title(f'Evolución temporal: {name}')
        plt.ylabel(name)
        plt.xlabel('Fecha')
        plt.legend(title='Estación', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()


### 2. Rangos de Voltaje en la Batería (Boxplots)
Comparamos la estabilidad de la batería. Cajas muy amplias indican grandes descargas de voltaje.

In [ ]:
if not df_diario.empty:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df_diario, x='station_name', y='battery_voltage_RANGE', palette='Set2')
    plt.title('Fluctuación del Voltaje de la Batería (Max - Min Diario)')
    plt.ylabel('Rango de Voltaje (V)')
    plt.xlabel('Estación')
    plt.show()


### 3. Matriz de Correlación de Variables Operativas
¿Qué impacta más en la generación de energía? ¿La temperatura del controlador? ¿El voltaje de la batería?

In [ ]:
if not df_diario.empty:
    variables_corr = [
        'pv_power_W_MEAN', 'battery_soc_%_MEAN', 'controller_temp_C_MEAN', 
        'energy_generated_today_kWh_MAX', 'battery_voltage_RANGE'
    ]
    
    for estacion in df_diario['station_name'].unique():
        df_est = df_diario[df_diario['station_name'] == estacion]
        
        corr_matrix = df_est[variables_corr].corr()
        
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True, vmin=-1, vmax=1)
        plt.title(f'Correlación - Estación: {estacion}')
        plt.tight_layout()
        plt.show()


## Exportación de la Capa Oro
Guardamos los datasets listos para ser consumidos por dashboards, reportes o modelos de Machine Learning predictivos.

In [ ]:
if not df_diario.empty and not df_base.empty:
    # Exportación a CSV
    path_oro_diario = ORO_DIR / 'dataset_oro_diario_solar.csv'
    path_oro_base = ORO_DIR / 'dataset_oro_horario_solar.csv'
    
    df_diario.to_csv(path_oro_diario, index=False)
    df_base.to_csv(path_oro_base, index=False)
    
    print('Archivos exportados exitosamente en la Capa Oro:')
    print(f' - {path_oro_diario}')
    print(f' - {path_oro_base}')


# Conclusión

En esta etapa final se construyó la **Capa Oro** del pipeline analítico para los sistemas Epever. 
Logramos:
1. **Creación de Variables de Valor (Feature Engineering):** Como el rango de fluctuación de voltaje o la categorización booleana de un 'día productivo'.
2. **Validación Exhaustiva:** Asegurando la nulidad neta (`0 NaN`) y continuidad temporal.
3. **Visualización Exploratoria:** Exponiendo patrones entre estaciones y correlaciones lógicas.
4. **Datasets Finales:** Tenemos un conjunto de datos robusto, veraz y listo para alimentar algoritmos de Clustering, Predicción de Series Temporales (ej. predecir fallas o descargas) o Reglas de Asociación.